# 03 | Foreign Key and Table Relationship Analysis

**Purpose:** Check whether key values in child tables match their referenced parent tables, and inspect geographic-prefix and product-category mapping gaps.

**Inputs:** The nine original Olist CSV files in `data/raw/`.

**Interpretation:** Unmatched values are data-quality or mapping findings to investigate; they are not automatically a reason to delete records. Geolocation zip-prefix checks are membership tests against the geolocation table, which can contain repeated prefixes.

Run after `01_dataset_audit.ipynb` and `02_primary_key_identifier_analysis.ipynb`.


In [ ]:
import pandas as pd
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = Path(os.environ.get("OLIST_DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser()

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Olist CSV folder not found: {DATA_DIR}\n"
        "Download the Olist CSV files and place them in data/raw, "
        "or set the OLIST_DATA_DIR environment variable."
    )

print("Dataset folder:", DATA_DIR)


## Load the source tables


In [ ]:
datasets = {}

for file in DATA_DIR.glob("*.csv"):
    datasets[file.stem] = pd.read_csv(file, low_memory=False)

print("Datasets loaded:", len(datasets))

## Validate key relationships between tables


In [ ]:
relationships = [
    ("olist_orders_dataset", "customer_id", "olist_customers_dataset", "customer_id"),
    ("olist_order_items_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_items_dataset", "product_id", "olist_products_dataset", "product_id"),
    ("olist_order_items_dataset", "seller_id", "olist_sellers_dataset", "seller_id"),
    ("olist_order_payments_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_reviews_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_products_dataset", "product_category_name", "product_category_name_translation", "product_category_name")
]

for child, col1, parent, col2 in relationships:
    a = datasets[child]
    b = datasets[parent]

    unmatched = a[col1].notna() & ~a[col1].isin(b[col2])

    print(child, "->", parent)
    print("Unmatched records:", unmatched.sum())
    print("-" * 40)

## Check customer and seller zip-prefix coverage


In [ ]:
geo = datasets["olist_geolocation_dataset"]

for table in ["olist_customers_dataset", "olist_sellers_dataset"]:
    df = datasets[table]

    unmatched = ~df["customer_zip_code_prefix"].isin(geo["geolocation_zip_code_prefix"]) if table == "olist_customers_dataset" else ~df["seller_zip_code_prefix"].isin(geo["geolocation_zip_code_prefix"])

    print(table, "Unmatched zip codes:", unmatched.sum())

In [ ]:
geo = datasets["olist_geolocation_dataset"]

for table, column in [
    ("olist_customers_dataset", "customer_zip_code_prefix"),
    ("olist_sellers_dataset", "seller_zip_code_prefix")
]:
    df = datasets[table]

    missing = df[
        ~df[column].isin(geo["geolocation_zip_code_prefix"])
    ]

    print(table, "Unmatched prefixes:", missing[column].nunique())

## Check product-category translation coverage


In [ ]:
products = datasets["olist_products_dataset"]

translation = datasets["product_category_name_translation"]

missing_categories = products[
    ~products["product_category_name"].isin(
        translation["product_category_name"]
    )
]

print("Unmatched product records:", len(missing_categories))

display(
    missing_categories["product_category_name"].value_counts(
        dropna=False
    )
)